# Modul 9: Feature Engineering Temporal & Machine Learning (XGBoost)
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 9 (Minggu 10):** Merumuskan peramalan beban sebagai *Supervised Learning* bebas *data leakage* (fitur lag, *rolling statistics*, *cyclical encoding* $\sin/\cos$) dan memvalidasinya dengan *TimeSeriesSplit*.

In [ ]:
# !pip install -q xgboost scikit-learn pandas numpy matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost belum terpasang; menggunakan HistGradientBoostingRegressor (scikit-learn) sebagai fallback.")

plt.rcParams['axes.grid'] = True

## 1. Pembangkitan Data Beban Multivariat & Rekayasa Fitur Kausal (Anti-Leakage)

In [ ]:
rng = np.random.default_rng(10)
n_hours = 24 * 60  # 60 hari
idx = pd.date_range('2026-01-01', periods=n_hours, freq='h')
hour = idx.hour
dow = idx.dayofweek

temp = 27 + 5 * np.sin(2 * np.pi * (hour - 8) / 24) + rng.normal(0, 1.2, n_hours)
load = (
    400 + 85 * np.sin(2 * np.pi * (hour - 6) / 24)
    + 60 * np.exp(-((hour - 19) ** 2) / 8)
    - 35 * (dow >= 5).astype(float)
    + 4.5 * temp
    + rng.normal(0, 7.0, n_hours)
)
df = pd.DataFrame({'load': load, 'temp': temp}, index=idx)

def create_causal_features(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()
    # Cyclical encoding jam & hari
    d['hour_sin'] = np.sin(2 * np.pi * d.index.hour / 24.0)
    d['hour_cos'] = np.cos(2 * np.pi * d.index.hour / 24.0)
    d['is_weekend'] = (d.index.dayofweek >= 5).astype(int)
    # Lag & Rolling kausal (WAJIB .shift(1) sebelum .rolling()!)
    d['lag_1'] = d['load'].shift(1)
    d['lag_24'] = d['load'].shift(24)
    d['lag_168'] = d['load'].shift(168)
    d['roll_mean_24'] = d['load'].shift(1).rolling(24).mean()
    d['roll_std_24'] = d['load'].shift(1).rolling(24).std()
    return d.dropna()

df_feat = create_causal_features(df)
feature_cols = ['temp', 'hour_sin', 'hour_cos', 'is_weekend', 'lag_1', 'lag_24', 'lag_168', 'roll_mean_24', 'roll_std_24']
X, y = df_feat[feature_cols], df_feat['load']
print(f"Matriks fitur kausal siap: X={X.shape}, y={y.shape}")

## 2. Validasi Expanding Window (TimeSeriesSplit) & Evaluasi Out-of-Sample

In [ ]:
tscv = TimeSeriesSplit(n_splits=4, test_size=168)
fold_rmses = []

for fold, (tr_idx, va_idx) in enumerate(tscv.split(X), 1):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]
    if HAS_XGB:
        model = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42)
    else:
        model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=5, random_state=42)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_va)
    rmse = np.sqrt(mean_squared_error(y_va, pred))
    mae = mean_absolute_error(y_va, pred)
    fold_rmses.append(rmse)
    print(f"Fold {fold} (Val 168 jam) -> RMSE: {rmse:.2f} MW | MAE: {mae:.2f} MW")

print(f"Rata-rata RMSE Expanding Window CV: {np.mean(fold_rmses):.2f} MW")

plt.figure(figsize=(12, 4.5))
plt.plot(y_va.index, y_va.values, 'k-', lw=1.8, label='Aktual (Fold Terakhir)')
plt.plot(y_va.index, pred, color='#10b981', lw=1.8, label='Prediksi ML')
plt.title('Short-Term Load Forecasting (168 Jam) dengan Rekayasa Fitur Temporal')
plt.ylabel('Beban (MW)')
plt.legend()
plt.show()

## 3. Tugas & Pertanyaan Analisis (C4–C6)
1. **[C4]** Hapus `.shift(1)` pada `roll_mean_24` lalu jalankan ulang. Mengapa RMSE turun secara artifisial (*data leakage*)?
2. **[C5]** Plot *Feature Importance* model XGBoost dan analisis mengapa `lag_24` dan `lag_168` sangat dominan pada beban listrik!